# Build a Full Text Preprocessing Pipeline

### The goal: combine everything you've practiced individually into one reusable function that takes raw, messy text and outputs clean, model-ready text — in the correct order

In [105]:
from string import punctuation

raw_texts = [
    "OMG!! $AAPL just surged 15%!! Can't believe it 😱 check https://finance.com/news?id=42",
    "I don't think the FED will raise rates... but I'm not sure tbh. Contact analyst@bank.com for more info.",
    "This stock's performance was sooooo disappointing, down -12% today. #investing #stocks",
    "Café Holdings Inc. reported earnings of $3.5 million in Q1 2024, beating estimates by 8%.",
    "@MarketWatch says the university's endowment fund isn't doing well. Definately a risky bet imo."
]

In [106]:
import unicodedata
import re
import contractions
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# For Tokenization
from nltk.tokenize import word_tokenize, sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\muham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\muham\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\muham\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


In [107]:
def preprocess_text(text):
    # 1 - Unicode Normalization
    text = unicodedata.normalize("NFC", text)

    # 2 - URL, email, and mention handling
    # Replacing URLS
    text = re.sub(r'https?://\S+', '<URL>', text)
    # Replacing Emails
    text = re.sub(r'\S+@\S+\.\S+', '<EMAIL>', text)
    # Replacing Mentions
    text = re.sub(r'@\w+', '<MENTION>', text)

    # 3 - Contraction Expansion
    text = contractions.fix(text)

    # 4 - Spelling Variations
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 5 - Sentence Tokenization
    # sent_tokenize(text) takes the single cleaned string and splits it into a list of sentences, using NLTK's built-in rules for recognizing sentence boundaries. Earlier, text was a single string, but now "sentences" is a list of strings. Now the output will be wrapped in a list, list containing one sentence
    sentences = sent_tokenize(text)

    # 6 - Word Tokenization - this tokenizes words within each sentence separately. This loops over every sentence in sentences list, and for each one, calls word_tokenize() to break it into individual word tokens. The result is a list of lists
    tokenized_sentences = [word_tokenize(sentence) for sentence in sentences]

    # 7 - Lowercasing. Now since I have a nested list, lowercasing needs to happen at the individual token level. I have to loop through each sentence's word list and lowercase each word
    # THe outer loop goes through each sentence's token list and the inner loop goes through each individual word within that sentence
    lowercased_sentences = [[word.lower() for word in sentence] for sentence in tokenized_sentences]

    # 8 - Punctuation removal
    punctuation_to_remove = set(string.punctuation) - {'$', '%'}
    no_punctuation_sentences = [[word for word in sentence if word not in punctuation_to_remove] for sentence in lowercased_sentences]

    # 9 - Number Handling (placeholder instead of deletion)
    number_handled_sentences = [
        [re.sub(r'\d+', '<NUM>', word) for word in sentence]
        for sentence in no_punctuation_sentences
    ]

    # 10 - Stopword Removal (preserving negation words)
    stop_words = set(stopwords.words('english'))
    negation_words = {"not", "no", "never", "n't"}
    custom_stopwords = stop_words - negation_words

    no_stopword_sentences = [
        [word for word in sentence if word not in custom_stopwords]
        for sentence in number_handled_sentences
    ]

    # 11 - Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized_sentences = [
        [lemmatizer.lemmatize(word) for word in sentence]
        for sentence in no_stopword_sentences
    ]

    return lemmatized_sentences
    return no_punctuation_sentences

In [108]:
# Loop over the list HERE — call the function once per string, not once for the whole list
cleaned_texts = [preprocess_text(t) for t in raw_texts]

In [109]:
# Xip takes two or more lists and pairs up their elements by position. Item 1 from list A with item 1 from list B
for original, cleaned in zip(raw_texts, cleaned_texts):
    print(f"Original: {original}")
    print(f"Cleaned: {cleaned}")
    print()

Original: OMG!! $AAPL just surged 15%!! Can't believe it 😱 check https://finance.com/news?id=42
Cleaned: [['omg'], ['$', 'aapl', 'surged', '<NUM>', '%'], ['not', 'believe', '😱', 'check', 'url']]

Original: I don't think the FED will raise rates... but I'm not sure tbh. Contact analyst@bank.com for more info.
Cleaned: [['not', 'think', 'fed', 'raise', 'rate', '..', 'not', 'sure', 'honest'], ['contact', 'email', 'info']]

Original: This stock's performance was sooooo disappointing, down -12% today. #investing #stocks
Cleaned: [['stock', "'s", 'performance', 'soo', 'disappointing', '-<NUM>', '%', 'today'], ['investing', 'stock']]

Original: Café Holdings Inc. reported earnings of $3.5 million in Q1 2024, beating estimates by 8%.
Cleaned: [['café', 'holding', 'inc.', 'reported', 'earnings', '$', '<NUM>.<NUM>', 'million', 'q<NUM>', '<NUM>', 'beating', 'estimate', '<NUM>', '%']]

Original: @MarketWatch says the university's endowment fund isn't doing well. Definately a risky bet imo.
Cleaned